---
title: Запуск SQL в JupyterLab
label: sql-in-jupyterlab
thumbnail: null
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
# subject: SQL Learning & Tooling
venue: GitHub & GitVerse Pages
abstract: |
  Настройка подключения и выполнение SQL-запросов в **JupyterLab**.
authors:
  - name: Alex Smirnov
    email: a9455440@gmail.com
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-11-12
abbreviations:
    MyST: Markedly Structured Text
    ПКМ: Правая Кнопка Мыши
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
    Pandas: Библиотека Python для анализа и обработки данных
    Polars: Мощный аналог Pandas на Rust/Python
---

После развертывания сервера MySQL и базы данных Sakila настроим рабочее пространство, чтобы сделать JupyterLab независимой автономной средой для работы с SQL.

(connection-init)=
## Создание подключения

Для подключения к базе данных Sakila и работы с ней в JupyterLab, создадим стартовую ячейку инициализации. Она связывает между собой Python, драйвер PyMySQL, библиотеку SQLAlchemy и расширение JupySQL, позволяя выполнять SQL-код прямо в ячейках блокнота:

In [1]:
# 1. Импорт библиотек
import pandas as pd
import sqlalchemy as sa
import sql

# 2. Настройка отображения таблиц в Pandas
pd.set_option('display.max_rows', 20)

# 3. Активация и настройка SQL-магии (JupySQL)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False
# %config SqlMagic.displaylimit = 20
# %config SqlMagic.feedback = False

# 4. Создание подключения через SQLAlchemy
connection_url = sa.engine.URL.create(
    drivername='mysql+pymysql',
    host='localhost',
    port=3306,
    database='sakila',
    username='root',
    password='********',
)
engine = sa.create_engine(connection_url)

# 5. Передача созданного движка подключения (Engine) в JupySQL
%sql engine

# 6. Контроль версий библиотек и статуса подключения
print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 20")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")

Pandas ver. 3.0.5: порог усечения строк уменьшен до 20
SQLAlchemy ver. 2.0.52: подключение создано
JupySQL ver. 0.11.1: подключен через SQLAlchemy Engine


(init-imports)=
### Разбор блока импортов

```python
# 1. Импорт библиотек
import pandas as pd
import sqlalchemy as sa
import sql
```

::::{glossary}
import pandas as pd
: :::{div}
  :class: text-sm
  - Импортируем библиотеку **Pandas** – инструмент для анализа табличных данных и работы с датафреймами (DataFrame).
  - **В контексте книги**: используем для усечения многострочного вывода результатов запроса: чтобы вывод, как в книге, отображал через многоточие только первые и последние пять строк вместо вывода всех.
  :::
::::

::::{glossary}
import sqlalchemy as sa
: :::{div}
  :class: text-sm
  - Импортируем библиотеку **SQLAlchemy** – промышленный стандарт Python для управления подключениями и работы с реляционными базами данных.
  - До 8-й главы делал иначе[^1]:
      ```python
      from sqlalchemy import create_engine
      from sqlalchemy.engine import URL
      ```
    Но потом захотел при подключении видеть версию SQLAlchemy и изменил.

    [^1]: Можно подсмотреть под спойлером [Notebook Cell](/adv-vscode/#hide-cell) в начале каждой из глав 2-7. Например, в [Главе 5](/ch05)
  :::
::::

::::{glossary}
import sql
: :::{div}
  :class: text-sm
  - Импортируем модуль **sql** библиотеки **JupySQL**[^2].
  - В коде используем только ради вывода версии JupySQL (`sql.__version__`).
  - Для работы самой SQL-магии (`%load_ext sql` и `%sql engine`) эта строка **не требуется** – расширение загружается ядром Jupyter независимо.
  
  [^2]: Несмотря на то, что библиотека при установке через Conda называется `jupysql`, разработчики библиотеки решили, что внутри Python ее корневой модуль для импорта будет называться коротко `sql` (так исторически повелось от ее предшественника `ipython-sql`).
  :::
::::

(init-details)=
### Разбор остальных блоков

Пояснить остальные блоки (так же, как предыдущий **Разбор блока импортов**) в будущем планирую. А пока поясню в комментариях:

:::{code} python
:filename: Ячейка инициализации и подключения к базе данных
# 1 —— Импорт библиотек ——————————————————————————————————————————————————————
import pandas as pd
import sqlalchemy as sa
import sql

# 2 —— Настройка отображения таблиц в Pandas —————————————————————————————————
# Снижаем порог усечения с дефолтных 60 до 20 строк
pd.set_option('display.max_rows', 20)

# 3 —— Активация и настройка SQL-магии (JupySQL) —————————————————————————————
# Загрузка расширения JupySQL в ядро блокнота
%load_ext sql
# Автоконвертация результата запроса в pandas.DataFrame
%config SqlMagic.autopandas = True
# Отключение служебной строки 'Running query in...'
%config SqlMagic.displaycon = False
# Не активен при autopandas=True; лимит (default = 10) передан в Pandas
# %config SqlMagic.displaylimit = 20
# Отключение счетчика затронутых строк 'X rows affected' (по умолчанию включен)
# %config SqlMagic.feedback = False

# 4 —— Создание подключения через SQLAlchemy —————————————————————————————————
# Сборка объекта URL с параметрами подключения к базе данных
connection_url = sa.engine.URL.create(
    drivername='mysql+pymysql',
    host='localhost',
    port=3306,
    database='sakila',
    username='root',
    password='********',
)
# Создание движка подключения SQLAlchemy Engine на основе сформированного URL
engine = sa.create_engine(connection_url)

# 5 —— Передача созданного движка подключения (Engine) в JupySQL —————————————
%sql engine

# 6 —— Контроль версий библиотек и статуса подключения ———————————————————————
print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 20")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")
:::

(postgres-connection)=
### Подключение к PostgreSQL

Механика SQLAlchemy универсальна: подключение к PostgreSQL настраивается точно так же, как и к используемой в проекте MySQL. Структура кода остаётся прежней, а **ключевое отличие** заключается лишь в диалекте, драйвере и номере порта.

Для перехода на PostgreSQL достаточно двух шагов:

1. Установить в {ref}`окружение <env-tools-connectors>` драйвер [psycopg2](https://www.psycopg.org/):

:::{code} bash
# (base) Anaconda Prompt
conda install -n ds-book --override-channels -c conda-forge -c defaults psycopg2
:::

2. В эталонном блоке параметров `URL.create()` заменить значение `drivername` с `mysql+pymysql` на `"postgresql+psycopg2"`, а порт – с `3306` на стандартный для PostgreSQL `5432` (если администратор базы не назначил другой).

Например, так выглядит блок подключения к учебной базе данных PostgreSQL в {ref}`бесплатном интенсиве Основы SQL от Simulative <bootcamp-intro>`:

:::{code} python
:filename: Блок подключения к базе данных PostgreSQL
# ... 1–3. Импорт библиотек, настройка Pandas и активация JupySQL ...

# 4. Создание подключения через SQLAlchemy
connection_url = sa.engine.URL.create(
    drivername="postgresql+psycopg2",     # диалект и драйвер PostgreSQL
    host="95.163.241.236",
    port=5432,                            # порт PostgreSQL (по умолчанию)
    database="simulative",
    username="student",
    password="qweasd963",
)
engine = sa.create_engine(connection_url)

# ... 5–6. Передача Engine в JupySQL (%sql engine) и контроль версий ...
:::

(run-sql-queries)=
## Выполнение SQL-запросов

**JupySQL** – инструмент, позволяющий писать и выполнять SQL-код прямо внутри ячеек блокнота Jupyter:

- **Интеграция**: превращает ячейку блокнота в SQL-редактор с помощью магических команд `%sql` и `%%sql`.
- **Удобство**: избавляет от необходимости писать вспомогательный код Python (ручное создание курсоров, методы `execute()` и `fetchall()`).
- **Связка с Pandas**: благодаря настройке `%config SqlMagic.autopandas = True` результат любого SQL-запроса сразу возвращается как `pandas.DataFrame`.
- **Поддержка СУБД**: через движок SQLAlchemy работает с любой базой данных (MySQL, PostgreSQL, SQLite и др.).

Чтобы выполнить SQL-запрос, достаточно написать в начале ячейки команду `%%sql`, ввести SQL-код и нажать {kbd}`Shift` + {kbd}`Enter`:

In [8]:
%%sql
SELECT
    customer_id,
    COUNT(*) AS tot_rentals,
    SUM(amount) AS tot_payments
FROM payment
GROUP BY customer_id;

599 rows affected.

,customer_id,tot_rentals,tot_payments
0,1,32,118.68
1,2,27,128.73
2,3,26,135.74
3,4,22,81.78
4,5,38,144.62
...,...,...,...
594,595,30,117.70
595,596,28,96.72
596,597,25,99.75
597,598,22,83.78


(sql-magic-formats)=
### Форматы %sql и %%sql

JupySQL различает два режима вызова:

1. Однострочный (`%sql`) – удобен для коротких служебных команд или однострочных запросов (в проекте не используется):

In [ ]:
%sql SELECT version();

2. Многострочный (`%%sql`) – превращает всю ячейку в полноценный SQL-редактор. При этом подсвечивает синтаксис **в блокноте** Jupyter[^3]:

[^3]: Трюк с подсветкой синтаксиса **на сайте** показан в разделе {ref}`Использование тегов на практике <using-cell-tags>`.

(highlight-magic-sql)=
:::{figure} media/jupysql-example-code-1240.png
:::

:::{caution} Правило первой строки для `%%sql`
Команда многострочной магии `%%sql` обязана стоять на самой первой строке ячейки.

Если поставить перед ней пустую строку или комментарий Python (`# ...`), интерпретатор Jupyter выдаст синтаксическую ошибку: многострочная магия обрабатывает только ячейки, которые явно начинаются с `%%`.
:::


---